In [35]:
print("Hello")

Hello


In [5]:
import os

os.system("pip install psutil gputil")


0

In [34]:
import psutil
import GPUtil
import platform

print("===== SYSTEM INFO =====")

# CPU Info
print("\n--- CPU ---")
print("Processor:", platform.processor())
print("Physical cores:", psutil.cpu_count(logical=False))
print("Total cores:", psutil.cpu_count(logical=True))
print("CPU Usage:", psutil.cpu_percent(interval=1), "%")

# RAM Info
print("\n--- RAM ---")
ram = psutil.virtual_memory()
print("Total RAM: {:.2f} GB".format(ram.total / (1024**3)))
print("Used RAM: {:.2f} GB".format(ram.used / (1024**3)))
print("RAM Usage:", ram.percent, "%")

# GPU Info
print("\n--- GPU ---")
gpus = GPUtil.getGPUs()
if gpus:
    for gpu in gpus:
        print("GPU Name:", gpu.name)
        print("GPU Memory Total:", gpu.memoryTotal, "MB")
        print("GPU Memory Used:", gpu.memoryUsed, "MB")
        print("GPU Load:", gpu.load * 100, "%")
else:
    print("No GPU detected")

print("\n===== DONE =====")

===== SYSTEM INFO =====

--- CPU ---
Processor: AMD64 Family 23 Model 49 Stepping 0, AuthenticAMD
Physical cores: 64
Total cores: 128
CPU Usage: 1.5 %

--- RAM ---
Total RAM: 255.87 GB
Used RAM: 89.07 GB
RAM Usage: 34.8 %

--- GPU ---
GPU Name: NVIDIA RTX A6000
GPU Memory Total: 49140.0 MB
GPU Memory Used: 33910.0 MB
GPU Load: 14.000000000000002 %
GPU Name: NVIDIA RTX A6000
GPU Memory Total: 49140.0 MB
GPU Memory Used: 29301.0 MB
GPU Load: 0.0 %

===== DONE =====


In [11]:
import numpy as np
import pandas as pd
import torch
import psutil
import wfdb
import ast
import os
import random
import shutil
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib
import seaborn as sns
from tqdm import tqdm
from scipy.signal import resample
from sklearn.metrics import f1_score, confusion_matrix, precision_recall_fscore_support

# Check installed versions
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Torch version:", torch.__version__)
print("Psutil version:", psutil.__version__)
print("WFDB version:", wfdb.__version__)
print("Matplotlib version:", matplotlib.__version__)  # Access version from matplotlib module
print("Seaborn version:", sns.__version__)
try:
    print("TQDM version:", tqdm.__version__)  # TQDM version
except AttributeError:
    print("TQDM version could not be found.")
print("Scipy version:", resample.__module__.split()[0])
print("Scikit-learn version:", f1_score.__module__.split()[0])


NumPy version: 2.0.2
Pandas version: 2.3.3
Torch version: 2.7.1+cpu
Psutil version: 7.2.2
WFDB version: 4.3.0
Matplotlib version: 3.10.8
Seaborn version: 0.13.2
TQDM version could not be found.
Scipy version: scipy.signal._signaltools
Scikit-learn version: sklearn.metrics._classification


In [36]:
import os
# Define the destination base path where files will be sorted according to their classes
destination_base = r'C:\Users\zen3Node\LEON\myenv\Data\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1\SortedByClass'
# Function to count the number of .dat and .hea files in each class folder
def count_files_in_class_folders(destination_base):
    print("\nCounting .dat and .hea files in each class folder:")
    for folder in os.listdir(destination_base):
        class_folder = os.path.join(destination_base, folder)
        if os.path.isdir(class_folder):
            # Count the number of .dat files in the class folder
            dat_files = [f for f in os.listdir(class_folder) if f.endswith('.dat')]
            # Count the number of .hea files in the class folder
            hea_files = [f for f in os.listdir(class_folder) if f.endswith('.hea')]
            print(f"{folder}: {len(dat_files)} .dat files, {len(hea_files)} .hea files")

# Call the function to count the files
count_files_in_class_folders(destination_base)



Counting .dat and .hea files in each class folder:
CD: 18166 .dat files, 18166 .hea files
HYP: 18166 .dat files, 18166 .hea files
MI: 18166 .dat files, 18166 .hea files
NORM: 18166 .dat files, 18166 .hea files
STTC: 18166 .dat files, 18166 .hea files


In [38]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda runtime:", torch.version.cuda)
print("gpu count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


torch: 2.7.1+cpu
cuda available: False
cuda runtime: None
gpu count: 0


CNN Normalized

In [ ]:
import os
import numpy as np
import wfdb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.signal import resample
from sklearn.metrics import f1_score, confusion_matrix, precision_recall_fscore_support

# =========================
# CONFIG
# =========================
sorted_dataset_path = r'C:\Users\zen3Node\LEON\myenv\Data\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1\SortedByClass'
class_folders = ['CD', 'HYP', 'MI', 'NORM', 'STTC']

TARGET_LEN = 500
BATCH_SIZE = 64
EPOCHS = 100
LR = 1e-3
DROPOUT = 0.5
SEED = 42

# Normalization mode:
#   "per_record" -> z-score per sample (per lead)
#   "dataset"    -> z-score using TRAIN-set mean/std (per lead)  ✅ recommended
NORMALIZATION_MODE = "dataset"

np.random.seed(SEED)
torch.manual_seed(SEED)

# =========================
# DATA LOADING
# =========================
def load_class_data(sorted_dataset_path, class_folders, target_length=500, fraction=1.0):
    data, labels, errors = [], [], []

    for label_idx, class_folder in enumerate(class_folders):
        folder_path = os.path.join(sorted_dataset_path, class_folder)
        files = [f for f in os.listdir(folder_path) if f.endswith('.dat')]

        num_files = int(len(files) * fraction)
        sampled_files = files[:num_files]

        for file in tqdm(sampled_files, desc=f"Loading {class_folder} data ({int(fraction * 100)}%)"):
            file_path = os.path.join(folder_path, file.replace('.dat', ''))
            try:
                signal, _ = wfdb.rdsamp(file_path)  # (T, leads)
                signal = signal.astype(np.float32)

                # Resample along time axis to target_length
                signal_resampled = resample(signal, target_length, axis=0).astype(np.float32)  # (target_length, leads)

                data.append(signal_resampled)
                labels.append(label_idx)
            except Exception as e:
                errors.append(file)
                print(f"Error loading {file}: {e}")

    if errors:
        print(f"Skipped {len(errors)} files due to errors.")

    X = np.array(data, dtype=np.float32)   # (N, T, C)
    y = np.array(labels, dtype=np.int64)   # (N,)
    return X, y

print("Loading 100% of the data from sorted dataset...")
X_data, y_labels = load_class_data(sorted_dataset_path, class_folders, target_length=TARGET_LEN, fraction=1.0)
print(f"Loaded {X_data.shape[0]} samples with shape {X_data.shape}")

# =========================
# TRAIN/TEST SPLIT
# =========================
print("Splitting data into training and test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_labels, test_size=0.2, random_state=SEED, stratify=y_labels
)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

# =========================
# NORMALIZATION
# =========================
def per_record_zscore(X, eps=1e-8):
    """
    Z-score each record independently, per lead.
    X: (N, T, C)
    """
    mean = X.mean(axis=1, keepdims=True)  # (N, 1, C)
    std = X.std(axis=1, keepdims=True)    # (N, 1, C)
    return (X - mean) / (std + eps)

def fit_dataset_zscore_params(X_train, eps=1e-8):
    """
    Fit mean/std on TRAIN set only, per lead.
    Returns mean/std shaped (1, 1, C) for broadcasting.
    X_train: (N, T, C)
    """
    mean = X_train.mean(axis=(0, 1), keepdims=True)  # (1, 1, C)
    std = X_train.std(axis=(0, 1), keepdims=True)    # (1, 1, C)
    std = np.maximum(std, eps)
    return mean.astype(np.float32), std.astype(np.float32)

def apply_dataset_zscore(X, mean, std):
    return (X - mean) / std

if NORMALIZATION_MODE == "per_record":
    print("Applying per-record z-score normalization (train & test independently)...")
    X_train = per_record_zscore(X_train)
    X_test = per_record_zscore(X_test)

elif NORMALIZATION_MODE == "dataset":
    print("Fitting dataset z-score params on TRAIN only, applying to train & test...")
    train_mean, train_std = fit_dataset_zscore_params(X_train)
    X_train = apply_dataset_zscore(X_train, train_mean, train_std)
    X_test = apply_dataset_zscore(X_test, train_mean, train_std)

else:
    raise ValueError("NORMALIZATION_MODE must be 'per_record' or 'dataset'")

# =========================
# PYTORCH DATASET / DATALOADER
# =========================
class ECGDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.float32),
            torch.tensor(self.y[idx], dtype=torch.long)
        )

train_dataset = ECGDataset(X_train, y_train)
test_dataset  = ECGDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# =========================
# MODEL
# =========================
class CNNModel(nn.Module):
    def __init__(self, input_dim, output_dim, dropout_rate=0.5, target_len=500):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(dropout_rate)
        self.relu = nn.ReLU()

        # After two pools: length becomes target_len / 4
        self.fc1_input_dim = 128 * (target_len // 4)
        self.fc1 = nn.Linear(self.fc1_input_dim, 128)
        self.fc2 = nn.Linear(128, output_dim)

    def forward(self, x):
        # x: (B, T, C) -> Conv1d expects (B, C, T)
        x = x.transpose(1, 2)
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.reshape(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# =========================
# TRAINING SETUP
# =========================
input_dim = X_train.shape[2]
output_dim = len(class_folders)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = CNNModel(input_dim=input_dim, output_dim=output_dim, dropout_rate=DROPOUT, target_len=TARGET_LEN).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

train_losses, test_losses = [], []
train_accuracies, test_accuracies = [], []
f1_scores = []

# =========================
# TRAIN LOOP
# =========================
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # ---- TRAIN
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in tqdm(train_loader, desc="Training"):
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct_train / total_train
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    print(f"Training loss: {train_loss:.4f}, Training Accuracy: {train_acc:.4f}")

    # ---- EVAL
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating"):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())

    test_loss /= len(test_loader)
    test_acc = correct / total
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, average=None, labels=np.arange(len(class_folders)), zero_division=0
    )
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted')
    f1_scores.append(weighted_f1)

    print(f"Validation loss: {test_loss:.4f}, Testing Accuracy: {test_acc:.4f}, Weighted F1 Score: {weighted_f1:.4f}")
    print("Classification Report:")
    for i, class_name in enumerate(class_folders):
        print(f"{class_name:5} - Precision: {precision[i]:.2f}, Recall: {recall[i]:.2f}, F1-score: {f1[i]:.2f}, Support: {support[i]}")

# =========================
# PLOTS
# =========================
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(test_losses, label="Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Test Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(test_accuracies, label="Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train vs Test Accuracy")
plt.legend()

plt.show()

# Confusion Matrix
conf_matrix = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_folders, yticklabels=class_folders)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# =========================
# SAVE MODEL
# =========================
model_save_path = r'C:\Users\zen3Node\LEON\myenv\Models\cnn_classification_model_normalized.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

# =========================
# FINAL METRICS
# =========================
print(f"\nFinal Weighted F1 Score: {f1_scores[-1]:.4f}")
print(f"Final Testing Accuracy: {test_accuracies[-1]:.4f}")
print(f"Final Testing Loss: {test_losses[-1]:.4f}")
print(f"Input Shape: {X_data.shape}")
